# 第7周 · 周五：AA实验与Peeking——实验系统的质检与最大的自我欺骗

> 难度：⭐⭐⭐⭐ | 定位：理解AA实验的质检价值和Peeking的危害，掌握序贯检验的基本概念
> 前置依赖：W7D1-D4（AB实验全流程） | 后续用途：实验平台建设、实验结果审核
> 面试地位：高级岗位面试高频题——"Peeking为什么不行？那怎么在中间看数据？"
> 配对课程：01python_learn W7D5（卡方与ANOVA）

---

## 📘 区块1：核心概念与第一性原理

### 1.1 AA实验——验证你的"随机化"是不是真的随机

AA实验是AB实验的"空白对照"版本：两组用户看到完全一样的东西（都是旧版），只是随机分成了两组。

**逻辑**：如果随机化是完美的 → 两组在实验前完全相同 → 实验后也不应该有差异。

**检测方法**：
```
跑100次AA实验，看p<0.05的比例。
- 如果≈5% → 随机化正确（p值在H₀为真时服从均匀分布）
- 如果远>5%（如20%）→ 分流系统有bug！
- 如果远<5%（如<1%）→ 可能p值计算有问题（过于保守）
```

**AA实验 = 实验系统的"单元测试"。** 没有通过AA验证的实验平台，跑出来的任何AB结果都不值得信。

### 1.2 Peeking——数据分析中最常见的"自我欺骗"

你每天看一眼p值——什么时候p<以下阈值，什么时候停止实验。听起来很合理？

**这样做会让实际的第一类错误率从5%飙升到20-30%！**

原因：p值会随数据积累而波动。就算H₀为真（策略无效），p值也可能在某个时刻偶然跌到0.05以下。如果你在这个时刻停止实验——你就在"挑选"偶然的低p值。

**直观理解**：假设你抛一枚公平硬币（H₀为真）。如果你抛10次就停，不太可能得到80%正面。但如果你抛1000次，中间几乎一定会经过一段"80%正面"的短暂窗口。Peeking就是在捕捉这些窗口。

### 1.3 从第一性原理推导：为什么Peeking会让α膨胀？

每次Peeking都是一次独立的假设检验。即使每次检验的α=0.05，累积的第一类错误率会远大于0.05。

```
如果你在实验过程中Peeking了k次，累积第一类错误率：
α_cumulative ≈ 1 - (1-α)^k（如果每次检验独立）

但实际上，Peeking的检验不是独立的（用的是累积数据），所以α膨胀比上面更复杂。
模拟结果表明：Peeking 5次 → α≈15%, Peeking 10次 → α≈20%, Peeking 20次 → α≈25%
```

**正确的做法**：事前定好样本量（基于MDE计算），达到样本量后一次性分析。不要中途看p值做决策。

### 1.4 如果必须中途查看——序贯检验（Sequential Testing）

序贯检验是为"中途查看"而设计的统计框架。它不是简单的"每看一次做一次t检验"，而是调整每次查看的显著性阈值，保证整体α控制在5%。

**序贯检验的关键思想**：
- 早期查看时用更严格的阈值（如p<0.001才算显著）
- 随着数据积累，阈值逐渐放宽
- 最终达到预定样本量时，阈值接近0.05

**常见方法**：
- Pocock边界：每次查看用相同的严格阈值
- O'Brien-Fleming边界：早期极严格，后期接近0.05
- Alpha-spending函数：预先"分配"α到每次中期分析

> **今天最重要的认知**：Peeking是数据分析师最常见的"自我欺骗"——你不是在"监控实验"，你是在"等待p值降到0.05以下"。正确的做法是事前定好样本量，跑够了再分析。如果必须中途查看，用序贯检验而不是反复做t检验。

### 1.5 面试高频考点

**面试官常问**：
- "什么是Peeking？为什么不能Peeking？"
- "AA实验是干什么的？怎么做？"
- "SRM是什么？怎么检测？"
- "如果PM每天问你要数据，你怎么办？"

**回答要点**：
- Peeking让α从5%飙升到20%+（在"挑选"偶然的低p值）
- AA实验验证分流系统的随机性（跑100次，看p<0.05是否≈5%）
- SRM用卡方检验检测分流比例是否与预期一致
- 给PM看数据完整性和护栏指标，但不给看主要指标的p值

## 🧩 区块2：知识点详细拆解

### 2.1 AA实验的完整操作流程

**为什么重要**：AA实验是新实验平台上线前的必做检查。

**操作步骤**：
1. 设置一个"AA实验"：两组配置完全相同
2. 让实验运行到足够样本量（至少与常规AB实验相同）
3. 对多个指标做假设检验
4. 统计p<0.05的指标比例
5. 重复以上步骤多次（如100次）
6. 看p<0.05的整体比例是否≈5%

**判断标准**：
```
100次AA实验中：
- p<0.05的次数在2-8次之间（2%-8%）→ 正常波动，随机化OK
- p<0.05的次数>10次（>10%）→ 分流系统有问题
- p<0.05的次数<1次（<1%）→ p值计算可能过于保守
```

**什么时候应该跑AA实验**：
- 新实验平台上线前
- 分流算法变更后
- 数据管道变更后
- 定期（如每月一次）作为持续监控

### 2.2 SRM（样本比例不匹配）——实验的分流bug检测

**为什么重要**：SRM是实验质量的第一道防线。

**定义**：你计划50:50分流，实际跑出来是48:52。如果这个差异统计显著，说明分流系统有问题。

**检测方法——卡方检验**：
```python
expected_control = total * 0.5  # 期望对照组人数
expected_treatment = total * 0.5  # 期望实验组人数
chi2 = (observed_control - expected_control)^2 / expected_control
     + (observed_treatment - expected_treatment)^2 / expected_treatment
p = 1 - chi2.cdf(chi2, 1)
```
如果p < 0.05（或更严格，p < 0.01），判定为SRM。

**SRM的常见原因**：
1. 分流哈希函数有bug
2. 实验组有严重bug导致用户被"自然排除"（如crash、加载失败）
3. 数据管道中某组数据被错误过滤
4. 机器人/爬虫流量没有被正确排除
5. 两组的数据上报率不同

**SRM是一条铁律**：SRM存在时，不要看任何其他指标的结果。先修bug，再重跑实验。

### 2.3 Peeking的数学模拟——眼见为实

**为什么重要**：光说Peeking有害没用，需要让PM和高管看到数据。

**让PM做这个思想实验**：
"假设策略完全无效（H₀为真），你每天看一次p值，连续看14天。只要有一天p<0.05你就宣布胜利。你觉得你'赢'的概率是多少？"

答案是约20-25%，而不是5%。

**为什么Peeking的诱惑如此强烈**：
- 好奇心："我就看看数据怎么样了"
- 压力："老板问结果怎么样了"
- 乐观偏误："万一是大效果呢，早点上线早受益"
- 沉没成本："已经跑了这么久，快看看有没有效果"

### 2.4 序贯检验（Sequential Testing）——合法的"中途查看"

**为什么重要**：在有些场景下（如医疗实验、高风险业务），确实需要在中间查看数据。序贯检验提供了统计上合法的框架。

**Alpha-spending方法**：
将总α=0.05分配到K次中期分析中：
```
分析1（1/3数据）: α₁ = 0.0001（极严格）
分析2（2/3数据）: α₂ = 0.01
分析3（全部数据）: α₃ = 0.04

总α = α₁ + α₂ + α₃ ≈ 0.05（近似，实际更复杂）
```

**O'Brien-Fleming边界**（最常用）：
- 第1次中期分析：p < 0.0001才算显著
- 第2次中期分析：p < 0.001才算显著
- 最终分析：p < 0.047才算显著（略低于0.05以补偿前几次查看）

**与naive Peeking的区别**：
```
Naive Peeking: 每次用p<0.05→α膨胀到20%+
Sequential: 每次用调整后的阈值→总α控制在5%
```

### 2.5 为什么业内人士仍然建议"不要Peeking"？

**为什么重要**：即使有统计学上合法的方法，实践中最安全的做法还是"不Peeking"。

**原因**：
1. **序贯检验的统计功效更低**：同样的样本量，序贯检验的Power低于固定样本设计。
2. **操作复杂度高**：需要精确按计划执行中期分析，错过一次计划时间点就会破坏α控制。
3. **解释困难**：PM很难理解"p=0.03为什么不显著？"（因为序贯阈值是0.01）。
4. **道德风险**：一旦开了"可以中途看"的口子，就容易滑向"随便看"。

**最佳实践**：
- 默认策略：固定样本量，到终点一次性分析
- 如果必须中期查看（如安全监控），只限于护栏指标
- 如果确实需要序贯检验，使用成熟的工具（而不是手动做t检验）

### 2.6 另一种"Peeking"——多指标Peeking

**为什么重要**：Peeking不只在时间维度上，也在指标维度上。

即使你只分析一次，但如果同时看20个指标，也会有问题：
```
20个指标，每个用α=0.05 → 至少1个误报的概率 ≈ 64%
```

**防御措施**：
- 预注册主要指标（1个），决策只看它
- 其他指标标注为"探索性"
- 如果必须评估多个指标，用多重比较校正

### 2.7 AA实验揭示了什么——经典案例

**为什么重要**：了解真实世界中AA实验能发现的问题。

**案例1：Microsoft的"一个月bug"**
Microsoft的实验团队曾发现，他们的实验平台在每月最后一天会出现分流错误——因为哈希函数的日期处理有bug。AA实验在这个特定日期跑出了25%的假阳性率。

**案例2：Google的"浏览器差异"**
Google曾发现某个实验在Chrome上SRM正常，但在Firefox上出现SRM——因为Firefox的一个bug导致部分用户无法进入实验组。AA实验按浏览器分层后发现了这个问题。

**案例3：Bot流量**
某电商的AA实验假阳性率高达30%。排查发现——大量爬虫流量被随机分配（爬虫的user_id是假的），这些爬虫的"转化行为"与真实用户完全不同，破坏了随机性。修复方法：排除非真实用户后再跑AA。

### 2.8 建立"实验诚信"文化

**为什么重要**：工具和技术最终服务于文化。没有诚信文化，再好的工具也没用。

**实验诚信的行为准则**：
1. 所有实验必须事前注册（假设、指标、样本量、决策规则）
2. 实验结果"显著"和"不显著"都必须报告
3. 不能事后修改主要指标或分析方法
4. AA实验和SRM检测的结果一并汇报
5. 实验数据库公开透明（所有利益相关方可查看）

**文化建设的关键**：
- 领导层要以身作则：不因"不显著"而惩罚团队
- 奖励"学习"而非"成功"：一个严谨设计的"失败实验" > 一个粗制滥造的"成功实验"
- 把"取消实验"和"否定自己假设"视为成熟分析师的标志

## 💻 区块3：动手练习

In [ ]:
import numpy as np
from scipy import stats
np.random.seed(42)

print("="*60)
print("练习0：AA实验——验证分流系统")
print("="*60)

# 模拟100次AA实验
n_sims = 100
n_users_per_group = 5000
baseline = 0.10

aa_p_values = []
for _ in range(n_sims):
    a1 = np.random.binomial(1, baseline, n_users_per_group)
    a2 = np.random.binomial(1, baseline, n_users_per_group)
    _, p = stats.ttest_ind(a1, a2)
    aa_p_values.append(p)

aa_p_values = np.array(aa_p_values)

print(f"模拟{n_sims}次AA实验（两组完全相同的配置）:")
print(f"  p<0.05的次数: {(aa_p_values < 0.05).sum()} / {n_sims}")
print(f"  比例: {(aa_p_values < 0.05).mean():.1%}")
print(f"  预期比例: ~5%")
print(f"  → {'✓ 分流正常' if 0.02 < (aa_p_values < 0.05).mean() < 0.08 else '✗ 需要排查！'}")
print(f"\n其他检查:")
print(f"  平均p值: {aa_p_values.mean():.3f} (H₀为真时预期≈0.5)")
print(f"  p值标准差: {aa_p_values.std():.3f} (H₀为真时预期≈0.29)")

In [ ]:
print("="*60)
print("练习1：Peeking的危害——模拟实验")
print("="*60)

def simulate_experiment_with_peeking(true_effect=0.0, n_per_day=500, 
                                      max_days=14, alpha=0.05):
    """模拟'每天看一次p值，显著就停'的策略"""
    control_data = []
    experiment_data = []
    p_history = []

    for day in range(1, max_days + 1):
        # 当天新增数据
        new_control = np.random.binomial(1, 0.10, n_per_day).tolist()
        new_experiment = np.random.binomial(1, 0.10 + true_effect, n_per_day).tolist()
        control_data.extend(new_control)
        experiment_data.extend(new_experiment)

        # 做检验
        _, p = stats.ttest_ind(control_data, experiment_data)
        p_history.append(p)

        if p < alpha:
            return day, p, True, p_history

    return max_days, p_history[-1], False, p_history

# 模拟1000次实验，策略完全无效（H₀为真）
n_sims = 1000
false_positives_peeking = 0
stopping_days = []

for _ in range(n_sims):
    day, _, stopped, _ = simulate_experiment_with_peeking(true_effect=0.0)
    if stopped:
        false_positives_peeking += 1
        stopping_days.append(day)

print(f"模拟{n_sims}次实验，策略完全无效（H₀为真）:")
print(f"\n【有Peeking（每天看p值，p<0.05就停）】")
print(f"  误报次数: {false_positives_peeking} / {n_sims}")
print(f"  误报率: {false_positives_peeking/n_sims:.1%}")
print(f"  预期（无Peeking）: ~5%")
print(f"  → Peeking让误报率从5%飙升到{false_positives_peeking/n_sims:.1%}！")

if stopping_days:
    print(f"\n  误报发生时的平均天数: {np.mean(stopping_days):.1f}天")
    print(f"  最早误报: 第{min(stopping_days)}天")
    print(f"  最晚误报: 第{max(stopping_days)}天")
    print(f"  → Peeking可以在任何时间点产生误报")

In [ ]:
print("="*60)
print("练习2：可视化一次Peeking的p值波动")
print("="*60)

np.random.seed(456)

# 模拟一次策略完全无效的实验，但每天记录p值
control = []
experiment = []
p_values_over_time = []

for day in range(1, 31):
    control.extend(np.random.binomial(1, 0.10, 500).tolist())
    experiment.extend(np.random.binomial(1, 0.10, 500).tolist())  # H₀为真
    _, p = stats.ttest_ind(control, experiment)
    p_values_over_time.append(p)

print("p值随天数变化（策略完全无效）:")
print(f"{'天数':<8} {'累计样本/组':<12} {'p值':<10} {'p<0.05?':<10}")
print("-" * 40)

for i, (day, p) in enumerate(zip(range(1, 31, 3), p_values_over_time[::3])):
    n = (i*3+1) * 500
    marker = '⚠ 误报！' if p < 0.05 else ''
    print(f"第{day:<5}天 {n:<12,} {p:<10.4f} {marker}")

# 找出p值低于0.05的天
low_p_days = [i+1 for i, p in enumerate(p_values_over_time) if p < 0.05]
if low_p_days:
    print(f"\np<0.05出现在: 第{low_p_days}天")
    print(f"→ 如果这些天停止，就会得到'显著'的假阳性结论！")
else:
    print(f"\n这次模拟没有出现p<0.05（但其他模拟中会出现）")

In [ ]:
print("="*60)
print("练习3：SRM模拟——分流比例偏差")
print("="*60)

np.random.seed(789)

# 模拟正确的分流和错误的分流
total_users = 10000

# 正确的50:50分流
correct_control = np.random.binomial(total_users, 0.5)
correct_treatment = total_users - correct_control

# 有bug的48:52分流
buggy_control = np.random.binomial(total_users, 0.48)
buggy_treatment = total_users - buggy_control

# SRM卡方检验
def srm_test(obs_control, obs_treatment, expected_ratio=0.5):
    total = obs_control + obs_treatment
    expected = total * expected_ratio
    chi2 = ((obs_control - expected)**2 / expected +
            (obs_treatment - expected)**2 / expected)
    p = 1 - stats.chi2.cdf(chi2, 1)
    return chi2, p

print("正确的50:50分流:")
chi2_correct, p_correct = srm_test(correct_control, correct_treatment)
print(f"  对照={correct_control}, 实验={correct_treatment}")
print(f"  卡方={chi2_correct:.4f}, p={p_correct:.4f}")
print(f"  → {'✗ SRM！' if p_correct < 0.05 else '✓ 通过（50:50的随机波动）'}")

print(f"\n有bug的48:52分流:")
chi2_buggy, p_buggy = srm_test(buggy_control, buggy_treatment)
print(f"  对照={buggy_control}, 实验={buggy_treatment}")
print(f"  卡方={chi2_buggy:.4f}, p={p_buggy:.4f}")
print(f"  → {'✗ SRM！分流有bug！' if p_buggy < 0.05 else '✓ 偶然通过（但比例本身就有问题）'}")

# 大样本下的SRM更容易被检测
print(f"\n大样本下SRM更容易检测（100万用户）:")
big_control = np.random.binomial(1000000, 0.48)
big_treatment = 1000000 - big_control
chi2_big, p_big = srm_test(big_control, big_treatment)
print(f"  对照={big_control}, 实验={big_treatment}")
print(f"  比例偏差={abs(big_control/1000000 - 0.5)*100:.2f}%")
print(f"  p={p_big:.2e} → SRM必定被检测（p极小）")
print(f"  → 大样本时即使是微小的分流偏差也会被检测出来")

In [ ]:
print("="*60)
print("挑战练习：序贯检验 vs Naive Peeking")
print("="*60)

np.random.seed(111)

# 简化版O'Brien-Fleming边界
def obrien_fleming_threshold(t, K=5, alpha=0.05):
    """t: 当前分析次数(1-indexed), K: 计划总分析次数"""
    # 简化的OBF阈值（实际应该基于alpha-spending）
    z_critical = 2.0 / np.sqrt(t / K)  # 近似
    return 2 * (1 - stats.norm.cdf(z_critical))  # 转回p值阈值

def simulate_sequential_vs_naive(true_effect, n_per_stage, K, n_sims=500):
    """对比序贯检验和Naive Peeking"""
    naive_fp = 0  # Naive Peeking的误报数
    seq_fp = 0    # 序贯检验的误报数

    for _ in range(n_sims):
        c_data = []
        e_data = []
        naive_stopped = False
        seq_stopped = False

        for stage in range(1, K + 1):
            c_data.extend(np.random.binomial(1, 0.10, n_per_stage).tolist())
            e_data.extend(np.random.binomial(1, 0.10 + true_effect, n_per_stage).tolist())
            _, p = stats.ttest_ind(c_data, e_data)

            # Naive: p<0.05就停
            if not naive_stopped and p < 0.05:
                naive_fp += 1
                naive_stopped = True

            # Sequential: 用调整后的阈值
            threshold = obrien_fleming_threshold(stage, K)
            if not seq_stopped and p < threshold:
                seq_fp += 1
                seq_stopped = True

    return naive_fp/n_sims, seq_fp/n_sims

naive_rate, seq_rate = simulate_sequential_vs_naive(
    true_effect=0.0, n_per_stage=2000, K=5, n_sims=500
)

print("策略完全无效（H₀为真）时，5次中期分析:")
print(f"  Naive Peeking误报率: {naive_rate:.1%}")
print(f"  序贯检验误报率: {seq_rate:.1%}")
print(f"  目标α: 5%")
print(f"\n→ Naive Peeking远超5%，序贯检验接近5%")
print(f"→ 但序贯检验的功效略低于固定样本设计")
print(f"→ 因此最佳实践仍然是：固定样本量，不Peeking")

## 📊 区块4：业务场景应用

### 场景1：PM每天问"显著了吗"——如何应对

**问题**：PM每天早上Slack你："实验显著了吗？"。你说"还不能看"。PM说"就看一眼，我们不基于它做决策"。

**分析**：这就是Peeking压力的经典场景。PM不是在恶意破坏实验——他只是不理解Peeking的危害。

**应对策略**：
1. **事前教育**：实验开始前和PM对齐——"我们会在X天后做一次性分析，中间不看p值。这不影响最终结论，反而让结论更可信。"
2. **给替代信息**：给PM看数据完整性和护栏指标——"数据采集正常，两组人数均衡，护栏指标OK"——这满足了他的"想知道进展"的需求，但不会破坏实验的统计有效性
3. **用类比解释**："就像烤蛋糕——你在烤的过程中频繁打开烤箱看，蛋糕反而烤不好。"
4. **建立规范**：在实验设计文档中明确写——"中期分析：仅查看数据完整性，不查看主要指标p值"

### 场景2：实验平台上线前的AA验证

**问题**：公司新建了AB实验平台，准备投入使用。你怎么验证它？

**分析**：在真实实验开始前，必须先跑AA实验验证分流系统。

**验证方案**：
1. 设置一个"空实验"——两组配置完全相同
2. 分配足够的流量（至少每组1万用户）
3. 对10个常用指标做假设检验
4. 重复10次（每次一周）
5. 统计p<0.05的比例
6. 如果比例显著偏离5%→排查原因

**排查清单（当AA失败时）**：
- [ ] 哈希函数是否正确（相同的user_id是否始终在同一组）
- [ ] 数据管道是否有bug（过滤、去重是否对两组一致）
- [ ] Bot流量是否被正确排除
- [ ] 时间窗口是否定义正确
- [ ] 两组用户特征是否均衡（不仅是人数，还有年龄、平台等）

### 场景3：连续3个实验都"不显著"——团队开始怀疑实验平台

**问题**：团队连续跑了3个实验，结果都不显著。PM开始质疑："是不是实验平台有问题？为什么一个成功的都没有？"

**分析**：
1. 实验"成功率"在业界是10-20%，连续3个不显著很正常
2. 但也要排除"假阴性"——是不是样本量不够（第二类错误）？
3. 以及排除平台问题——跑一个AA实验验证

**应对**：
1. 跑AA实验——如果通过，说明平台没问题
2. 回顾每个实验的MDE和实际样本量——如果样本量不够，不显著是正常的
3. 向团队教育："不显著≠实验失败。你排除了三个不work的方向，节省了全量上线的成本。"
4. 看效应量和置信区间——也许有一些"方向性"信号可以作为下一步优化的输入

### 场景4：高管说"我在上一家公司都是天天看数据的"

**问题**：新来的VP说："我在上一家公司，每个实验都有实时dashboard，我们天天看数据，看到好的就上线。你们为什么这么保守？"

**分析**：这是一种常见的"经验偏差"——有些公司的"实验"其实质量管理非常粗糙，通过数量掩盖质量。

**回应框架**：
1. 不否定VP的经验："实时dashboard是有价值的——用于监控数据质量和护栏指标。"
2. 解释Peeking的科学依据："统计研究表明，每天看p值做决策会让误报率从5%升到20%+。相当于我们每上线5个'显著'的策略，有1个实际是无效的。"
3. 提供折中方案："我们可以提供实时监控dashboard，但只看数据质量和护栏指标。主要指标的p值在达到预定样本量后再公布。"
4. 邀请参与："如果您感兴趣，我们可以跑一次AA实验，让你亲眼看到——即使策略完全无效，p值也会随机地低于0.05。"

## ✅ 区块5：分级检验清单

### 🟢 初级检验（必须能独立完成的）
- [ ] 能解释AA实验是什么以及为什么需要
- [ ] 能解释Peeking为什么会导致误报率上升
- [ ] 能说出SRM的检测方法（卡方检验）
- [ ] 能说出"不Peeking"的正确做法

**答案提示**：
- AA实验：两组配置完全相同，验证分流系统的随机性
- Peeking在"挑选"偶然的低p值，累积α远超0.05
- SRM用卡方检验检测：如果p<0.05，分流有问题
- 正确做法：事前定样本量，到终点一次性分析

### 🟡 中级检验（能解释给同事听的原理）
- [ ] 能设计AA实验的验证方案
- [ ] 能用模拟数据演示Peeking的危害
- [ ] 能解释序贯检验和Naive Peeking的区别
- [ ] 能处理PM和高管的"Peeking压力"

**答案提示**：
- AA验证：跑100次AA→统计p<0.05比例→应≈5%
- 模拟：策略无效+每天Peeking→误报率20%+
- 序贯检验用调整后的阈值，总α控制在5%
- 应对Peeking压力：给数据完整性而非p值，事前约定好分析时间

### 🔴 高级检验（能举一反三的迁移能力）
- [ ] 能为实验平台设计AA验证和SRM检测的自动化流程
- [ ] 能选择合适的序贯检验方案（Pocock vs OBF vs Alpha-spending）
- [ ] 能建立团队的实验诚信文化
- [ ] 能在"必须中途查看"的约束下设计统计上合法的方案

**答案提示**：
- 自动化：每日自动SRM检查+护栏告警+AA定期跑
- Pocock边界不变、OBF早期严格后期宽松、Alpha-spending灵活分配
- 诚信文化：事前注册、所有结果公开、奖励学习而非成功
- 中途查看：用序贯检验框架，使用专业工具而非手动t检验

## 🔗 区块6：跨模块连接

| 连接方向 | 关联内容 | 为什么有关 |
|---------|---------|----------|
| **← 前置依赖** | W7D2 假设检验 | 理解p值的行为是理解Peeking的前提 |
| **← 前置依赖** | W7D4 实验设计 | 固定样本量设计是对抗Peeking的基础 |
| **→ 后续用途** | W7D6 综合实战 | 实际实验时需要AA验证和防Peeking |
| **→ 后续用途** | W8 序贯检验进阶 | 更深入的序贯检验方法 |
| **业务连接** | 实验平台建设 | AA和SRM检测是实验平台的基础功能 |
| **业务连接** | 数据分析文化 | 诚信文化是数据分析师职业操守的核心 |

### 横向连接：与其他分析方法的关联

**与数据质量监控的连接**：
- AA实验本质是一种数据质量监控
- SRM检测是每个实验的必做检查
- 护栏监控是业务层面的数据质量

**与软件工程的连接**：
- 分流系统需要单元测试和集成测试
- AA实验是分流系统的端到端测试
- Bot检测和过滤是数据工程质量的一部分

## 📝 区块7：当日小结

> **AA实验是实验系统的"单元测试"——跑100次，p<0.05的比例应≈5%，否则分流系统有bug。Peeking是数据分析师最常见的自我欺骗——每天看p值并在p<0.05时停止，会让实际误报率从5%飙升到20%+。正确的做法是事前定好样本量，跑够了再分析。如果必须中途查看，需要用序贯检验（调整每次查看的显著性阈值），而不是反复做普通的t检验。SRM用卡方检验检测——存在SRM时不要看任何其他指标，先修bug。**

**三条金句带回家**：
1. Peeking让α从5%飙升到20%+——你不是在"监控实验"，你是在"等待p值降到0.05以下"
2. AA实验是实验系统的"单元测试"——没通过AA的平台，任何AB结果都不值得信
3. SRM是实验的"红灯"——一票否决，不需要看任何其他指标

### 🔖 课后拓展

1. **练习**：写一个Python脚本，接收实验的每日数据流，自动进行SRM检测和护栏指标监控（但不输出主要指标p值）。这是"合法监控"的实践。

2. **思考**：如果一个实验需要跑8周（因为留存是主要指标），期间PM可能换人、产品方向可能调整，你有什么策略应对这种"组织风险"？

3. **预习**：明天综合实战——用这周学的所有知识，端到端完成一个AB实验。

4. **面试准备**：背诵以下面试题答案：
   - "什么是Peeking？为什么不行？"
   - "AA实验怎么做？有什么用？"
   - "SRM怎么检测？发现SRM后怎么办？"

---

*03analyst_thinking · Week 7 Day 5 · 2026-05-21*